# DGFRL: Disagreement-Guided Federated Representation Learning
## Full runnable PyTorch notebook scaffold based on the uploaded manuscript

This notebook implements the manuscript's main pipeline:
1. Three local image domains/clients (CheXpert, MIMIC-CXR-JPG, PadChest).
2. Patient-separated splits and multi-label harmonisation.
3. Hybrid local/global patch encoder (CNN patch stem + lightweight state-space/Mamba-inspired branch + window-attention Transformer branch).
4. DGMRL self-supervised learning: teacher/student views, patch disagreement, disagreement-guided plus random masking, latent prediction and consistency loss.
5. SRPA: feature-stability/branch-agreement routing, shared low-rank adapters and client-private adapters.
6. Simulated federated rounds, protected client updates with per-patient gradient clipping/noise, and weighted shared-parameter aggregation.
7. Supervised multi-label classification and evaluation metrics.

**Data access note:** MIMIC-CXR-JPG requires PhysioNet access and agreement. CheXpert and PadChest must be obtained from their official sources. This notebook does not download restricted datasets or invent results. Set the dataset paths and metadata column names in the configuration cell. A smoke-test mode creates synthetic images only to verify the code path; do not use it for scientific results.

**Privacy caveat:** the included patient-level clipping/noise is an educational DP-SGD-style mechanism, not a complete formal privacy guarantee. A publishable epsilon/delta requires an accountant matched to the exact patient-sampling procedure and all communicated data-dependent parameters. Secure aggregation is represented by server-side aggregation only; production secure aggregation requires a cryptographic protocol.

## 1. Install/import dependencies

In [ ]:
# In Colab, uncomment if needed:
# !pip -q install torch torchvision pandas scikit-learn pillow tqdm

import os, math, copy, random, json, time, warnings
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Configuration and dataset metadata contract

In [ ]:
# Edit these paths to your locally downloaded datasets.
# Each metadata CSV must contain: image_path, patient_id, and the finding columns below.
DATASETS = {
    "chexpert": {"csv": "/content/data/chexpert/metadata.csv", "image_root": "/content/data/chexpert"},
    "mimic":    {"csv": "/content/data/mimic/metadata.csv",    "image_root": "/content/data/mimic"},
    "padchest": {"csv": "/content/data/padchest/metadata.csv", "image_root": "/content/data/padchest"},
}
IMAGE_COL = "image_path"
PATIENT_COL = "patient_id"
LABELS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
IMG_SIZE = 224
BATCH_SIZE = 8
NUM_WORKERS = 2
PATCH_SIZE = 16
EMBED_DIM = 128
DEPTH = 2
NUM_HEADS = 4
MASK_RATIO = 0.40
DISAGREEMENT_MIX = 0.60
TEACHER_MOMENTUM = 0.996
LORA_RANK = 8
MIN_SHARED_GATE = 0.20
DP_CLIP_NORM = 1.0
DP_NOISE_MULTIPLIER = 0.5
FED_ROUNDS = 2
LOCAL_SSL_EPOCHS = 1
SUPERVISED_EPOCHS = 3
MAX_IMAGES_PER_CLIENT = None  # e.g. 500 for a quick pipeline test; None uses all rows
USE_SYNTHETIC_SMOKE_TEST = False  # True verifies execution only; not valid evaluation data

### Expected CSV format

In [ ]:
# Example only; do not execute unless you want to inspect the schema.
example = pd.DataFrame({
    "image_path": ["images/patient1_img1.jpg", "images/patient2_img1.jpg"],
    "patient_id": ["patient1", "patient2"],
    "Atelectasis": [1, 0], "Cardiomegaly": [0, 1],
    "Consolidation": [0, 0], "Edema": [np.nan, 0], "Pleural Effusion": [1, 0]
})
display(example)
print("Labels are multi-label binary targets. Use NaN for unknown/uncertain labels; they are masked from the loss.")

## 3. Metadata loading, label harmonisation, patient-level splits, and local datasets

In [ ]:
def make_smoke_data(root="/tmp/dgfrl_smoke", n=90, clients=("chexpert","mimic","padchest")):
    # Synthetic data is only for a software smoke test.
    root = Path(root); root.mkdir(parents=True, exist_ok=True)
    paths = {}
    for ci, name in enumerate(clients):
        folder = root / name; folder.mkdir(parents=True, exist_ok=True)
        rows = []
        for i in range(n):
            arr = np.random.default_rng(SEED + ci*10000 + i).integers(0, 256, (IMG_SIZE, IMG_SIZE), dtype=np.uint8)
            p = folder / f"img_{i}.png"; Image.fromarray(arr).save(p)
            row = {IMAGE_COL: str(p), PATIENT_COL: f"{name}_patient_{i//2}"}
            for lab in LABELS: row[lab] = int(np.random.rand() > 0.7)
            rows.append(row)
        csv = folder / "metadata.csv"; pd.DataFrame(rows).to_csv(csv, index=False)
        paths[name] = {"csv": str(csv), "image_root": str(folder)}
    return paths

def load_client_frame(name, cfg):
    csv_path = Path(cfg["csv"])
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Missing metadata CSV for client '{name}': {csv_path}\n"
            "Download/prepare the dataset and create a CSV with image_path, patient_id, and label columns. "
            "Or set USE_SYNTHETIC_SMOKE_TEST=True to test the pipeline."
        )
    df = pd.read_csv(csv_path)
    required = [IMAGE_COL, PATIENT_COL] + LABELS
    missing = [c for c in required if c not in df.columns]
    if missing: raise ValueError(f"{name}: missing CSV columns {missing}")
    root = Path(cfg.get("image_root", "."))
    def resolve(p):
        p = Path(str(p))
        return str(p if p.is_absolute() else root / p)
    df[IMAGE_COL] = df[IMAGE_COL].map(resolve)
    df = df[df[IMAGE_COL].map(lambda p: Path(p).is_file())].copy()
    if MAX_IMAGES_PER_CLIENT: df = df.head(MAX_IMAGES_PER_CLIENT).copy()
    for c in LABELS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df.loc[~df[c].isin([0,1]), c] = np.nan
    df = df.drop_duplicates(subset=[PATIENT_COL, IMAGE_COL]).reset_index(drop=True)
    if df.empty: raise ValueError(f"{name}: no readable image files after path filtering.")
    return df

def patient_split(df, seed=SEED, train_frac=0.70, val_frac=0.15):
    patients = df[PATIENT_COL].astype(str).drop_duplicates().to_numpy()
    rng = np.random.default_rng(seed); rng.shuffle(patients)
    n_train = max(1, int(len(patients)*train_frac))
    n_val = int(len(patients)*val_frac)
    train_p = set(patients[:n_train]); val_p = set(patients[n_train:n_train+n_val])
    train = df[df[PATIENT_COL].astype(str).isin(train_p)].copy()
    val = df[df[PATIENT_COL].astype(str).isin(val_p)].copy()
    test = df[~df[PATIENT_COL].astype(str).isin(train_p | val_p)].copy()
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

if USE_SYNTHETIC_SMOKE_TEST:
    DATASETS = make_smoke_data()

CLIENT_FRAMES, CLIENT_SPLITS = {}, {}
for client_name, cfg in DATASETS.items():
    frame = load_client_frame(client_name, cfg)
    splits = patient_split(frame)
    CLIENT_FRAMES[client_name] = frame
    CLIENT_SPLITS[client_name] = {"train": splits[0], "val": splits[1], "test": splits[2]}
    print(client_name, {k: len(v) for k,v in CLIENT_SPLITS[client_name].items()},
          "| unique patients:", frame[PATIENT_COL].nunique())

In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, frame, train=False, img_size=IMG_SIZE):
        self.df = frame.reset_index(drop=True)
        self.train = train; self.img_size = img_size
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row[IMAGE_COL]).convert("L")
        # Aspect-ratio-preserving resize then center pad.
        w,h = img.size; scale = self.img_size / max(w,h)
        nw, nh = max(1,int(w*scale)), max(1,int(h*scale))
        img = img.resize((nw,nh), Image.Resampling.BILINEAR)
        canvas = Image.new("L", (self.img_size,self.img_size), color=0)
        canvas.paste(img, ((self.img_size-nw)//2, (self.img_size-nh)//2))
        if self.train:
            if random.random() < 0.5: canvas = ImageEnhance.Contrast(canvas).enhance(random.uniform(0.9,1.1))
            if random.random() < 0.5: canvas = ImageEnhance.Brightness(canvas).enhance(random.uniform(0.95,1.05))
        x = np.asarray(canvas, dtype=np.float32)/255.0
        # Per-image percentile normalisation.
        lo,hi = np.percentile(x,[1,99]); x = np.clip((x-lo)/max(float(hi-lo),1e-6),0,1)
        x = torch.from_numpy(x[None,...]).float()
        y = row[LABELS].to_numpy(dtype=np.float32)
        mask = np.isfinite(y).astype(np.float32)
        y = np.nan_to_num(y, nan=0.0)
        return x, torch.tensor(y), torch.tensor(mask), str(row[PATIENT_COL])

def make_loader(frame, train, batch_size=BATCH_SIZE, shuffle=None):
    ds = ChestXrayDataset(frame, train=train)
    return DataLoader(ds, batch_size=batch_size, shuffle=train if shuffle is None else shuffle,
                      num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                      persistent_workers=NUM_WORKERS > 0)

## 4. Hybrid local/global encoder and SRPA low-rank adapters

In [ ]:
class LowRankAdapter(nn.Module):
    def __init__(self, dim, rank=LORA_RANK):
        super().__init__()
        self.A = nn.Linear(dim, rank, bias=False)
        self.B = nn.Linear(rank, dim, bias=False)
        nn.init.normal_(self.A.weight, std=0.02); nn.init.zeros_(self.B.weight)
        self.scale = 1.0 / max(rank,1)
    def forward(self, x): return self.B(self.A(x)) * self.scale

class SSMInspiredBlock(nn.Module):
    # Lightweight selective state-space-inspired sequence block, no custom CUDA Mamba dependency.
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.in_proj = nn.Linear(dim, 2*dim)
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=5, padding=2, groups=dim)
        self.out_proj = nn.Linear(dim, dim)
        self.gate = nn.Linear(dim, dim)
    def forward(self, x):
        z, g = self.in_proj(self.norm(x)).chunk(2, dim=-1)
        z = self.dwconv(z.transpose(1,2)).transpose(1,2)
        z = F.silu(z) * torch.sigmoid(g)
        return x + self.out_proj(z)

class HybridBlock(nn.Module):
    def __init__(self, dim=EMBED_DIM, heads=NUM_HEADS):
        super().__init__()
        self.ssm = SSMInspiredBlock(dim)
        self.attn_norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.ff_norm = nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim,4*dim), nn.GELU(), nn.Linear(4*dim,dim))
        self.shared_adapter = LowRankAdapter(dim)
        self.private_adapter = LowRankAdapter(dim)
        self.route = nn.Sequential(nn.Linear(2,32), nn.GELU(), nn.Linear(32,1), nn.Sigmoid())
    def forward(self, x, stability=None, disagreement=None, use_private=True):
        x = self.ssm(x)
        a,_ = self.attn(self.attn_norm(x), self.attn_norm(x), self.attn_norm(x), need_weights=False)
        x = x + a
        # If routing signals are absent, default to minimum shared contribution.
        if stability is None: stability = x.new_ones((x.shape[0],x.shape[1],1))
        if disagreement is None: disagreement = x.new_zeros((x.shape[0],x.shape[1],1))
        gate = MIN_SHARED_GATE + (1-MIN_SHARED_GATE)*self.route(torch.cat([stability, disagreement],dim=-1))
        adapted = gate*self.shared_adapter(x)
        if use_private: adapted = adapted + (1-gate)*self.private_adapter(x)
        x = x + adapted
        return x + self.ff(self.ff_norm(x)), gate

class HybridEncoder(nn.Module):
    def __init__(self, img_size=IMG_SIZE, patch_size=PATCH_SIZE, dim=EMBED_DIM, depth=DEPTH):
        super().__init__()
        self.patch_size=patch_size; self.dim=dim
        self.patch_embed = nn.Conv2d(1,dim,kernel_size=patch_size,stride=patch_size)
        n=(img_size//patch_size)**2
        self.pos = nn.Parameter(torch.zeros(1,n,dim))
        nn.init.trunc_normal_(self.pos,std=0.02)
        self.blocks=nn.ModuleList([HybridBlock(dim) for _ in range(depth)])
        self.norm=nn.LayerNorm(dim)
        self.local_projection=nn.Linear(dim,dim)
        self.global_projection=nn.Linear(dim,dim)
    def forward(self, x, stability=None, disagreement=None, use_private=True):
        t=self.patch_embed(x).flatten(2).transpose(1,2)
        t=t+self.pos[:,:t.shape[1]]
        # Parallel local and global feature views.
        local=self.local_projection(t)
        glob=t
        gates=[]
        for block in self.blocks:
            glob,gate=block(glob,stability,disagreement,use_private)
            gates.append(gate)
        fused=self.norm(local+glob)
        return fused, local, glob, gates

class DGFRLModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=HybridEncoder()
        self.predictor=nn.Sequential(nn.Linear(EMBED_DIM,EMBED_DIM),nn.GELU(),nn.Linear(EMBED_DIM,EMBED_DIM))
        self.classifier=nn.Sequential(nn.LayerNorm(EMBED_DIM),nn.Linear(EMBED_DIM,len(LABELS)))
    def forward(self,x,stability=None,disagreement=None,use_private=True):
        fused,local,glob,gates=self.encoder(x,stability,disagreement,use_private)
        pooled=fused.mean(dim=1)
        logits=self.classifier(pooled)
        return {"tokens":fused,"local":local,"global":glob,"pooled":pooled,"logits":logits,"gates":gates}

def make_model(): return DGFRLModel().to(DEVICE)
model = make_model()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. DGMRL: disagreement-guided masking, teacher, prediction, consistency

In [ ]:
@torch.no_grad()
def update_teacher(teacher, student, momentum=TEACHER_MOMENTUM):
    sdict=student.state_dict()
    for name, tp in teacher.named_parameters():
        sp=dict(student.named_parameters())[name]
        tp.data.mul_(momentum).add_(sp.data, alpha=1-momentum)
    # Copy buffers such as positional embeddings exactly.
    for name,tb in teacher.named_buffers():
        sb=dict(student.named_buffers()).get(name)
        if sb is not None: tb.copy_(sb)

def make_mask(disagreement, mask_ratio=MASK_RATIO, mix=DISAGREEMENT_MIX):
    # disagreement: [B,N], fixed number masked, weighted sampling without replacement.
    B,N=disagreement.shape; nmask=max(1,min(N-1,int(N*mask_ratio)))
    masks=[]
    for b in range(B):
        d=disagreement[b].detach().float()
        d=(d-d.min())/(d.max()-d.min()+1e-6)
        weights=mix*(d+1e-3)+(1-mix)*torch.ones_like(d)
        idx=torch.multinomial(weights, nmask, replacement=False)
        m=torch.zeros(N,device=d.device,dtype=torch.bool); m[idx]=True; masks.append(m)
    return torch.stack(masks)

def masked_tokens(tokens, mask):
    return tokens.masked_fill(mask.unsqueeze(-1), 0.0)

def dg_mrl_step(student, teacher, images, optimizer):
    student.train(); teacher.eval()
    with torch.no_grad():
        # Teacher sees the unmasked view.
        tout=teacher(images, use_private=False)
        tlocal=F.normalize(tout["local"],dim=-1)
        tglobal=F.normalize(tout["global"],dim=-1)
        disagreement=(1-F.cosine_similarity(tlocal,tglobal,dim=-1)).clamp(min=0)
        mask=make_mask(disagreement)
        target=F.normalize((tlocal+tglobal)/2,dim=-1)
    # Student encoder runs on view and masked patch tokens.
    s_local_img=student.encoder.patch_embed(images).flatten(2).transpose(1,2)
    s_local_img=s_local_img+student.encoder.pos[:,:s_local_img.shape[1]]
    s_local=student.encoder.local_projection(s_local_img)
    masked=masked_tokens(s_local,mask)
    # Pass masked tokens through predictor; consistency on visible tokens and prediction on masked.
    pred=student.predictor(masked)
    pred=F.normalize(pred,dim=-1)
    masked_loss=((pred-target)**2).mean(dim=-1)[mask].mean()
    # Full student branch consistency, encouraging complementary local/global representations.
    sout=student(images, use_private=False)
    sl=F.normalize(sout["local"],dim=-1); sg=F.normalize(sout["global"],dim=-1)
    consistency=(1-F.cosine_similarity(sl,sg,dim=-1)).mean()
    loss=masked_loss+0.2*consistency
    optimizer.zero_grad(set_to_none=True); loss.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    optimizer.step()
    update_teacher(teacher,student)
    return float(loss.detach()), float(mask.float().mean().detach())

teacher = copy.deepcopy(model).to(DEVICE)
for p in teacher.parameters(): p.requires_grad_(False)
print("Student/teacher ready.")

## 6. Patient-level clipped/noisy local updates and federated aggregation

In [ ]:
def get_shared_state(model):
    # Share all parameters except private adapter factors and private classification is shared here.
    state={}
    for k,v in model.state_dict().items():
        if ".private_adapter." not in k:
            state[k]=v.detach().cpu().clone()
    return state

def load_shared_state(model, shared_state):
    current=model.state_dict()
    for k,v in shared_state.items():
        if k in current and ".private_adapter." not in k:
            current[k]=v.to(current[k].device)
    model.load_state_dict(current, strict=True)

def patient_grouped_dp_step(model, optimizer, batch, clip_norm=DP_CLIP_NORM, noise_multiplier=DP_NOISE_MULTIPLIER):
    # Approximate patient-level DP-SGD: per-patient loss, clip each patient gradient, add Gaussian noise.
    images, labels, label_mask, patient_ids = batch
    images=images.to(DEVICE); labels=labels.to(DEVICE); label_mask=label_mask.to(DEVICE)
    optimizer.zero_grad(set_to_none=True)
    params=[p for p in model.parameters() if p.requires_grad]
    patient_ids=list(patient_ids)
    unique=list(dict.fromkeys(patient_ids))
    accum=[torch.zeros_like(p) for p in params]
    for pid in unique:
        ix=[i for i,p in enumerate(patient_ids) if p==pid]
        out=model(images[ix],use_private=True)["logits"]
        raw=F.binary_cross_entropy_with_logits(out,labels[ix],reduction="none")
        loss=(raw*label_mask[ix]).sum()/(label_mask[ix].sum()+1e-8)
        grads=torch.autograd.grad(loss,params,retain_graph=False,allow_unused=True)
        norm=torch.sqrt(sum((g.detach()**2).sum() for g in grads if g is not None)+1e-12)
        scale=min(1.0, clip_norm/(float(norm)+1e-12))
        for j,g in enumerate(grads):
            if g is not None: accum[j].add_(g.detach(),alpha=scale)
    denom=max(1,len(unique))
    for p,a in zip(params,accum):
        noise=torch.randn_like(a)*(noise_multiplier*clip_norm)
        p.grad=(a+noise)/denom
    optimizer.step()
    return len(unique)

def federated_average(client_states, weights=None):
    if not client_states: raise ValueError("No client states to aggregate.")
    if weights is None: weights=[1.0]*len(client_states)
    weights=np.asarray(weights,dtype=np.float64); weights=weights/weights.sum()
    out={}
    for k in client_states[0]:
        vals=[s[k] for s in client_states]
        if torch.is_floating_point(vals[0]):
            out[k]=sum(v*w for v,w in zip(vals,weights))
        else: out[k]=vals[0].clone()
    return out

## 7. Federated DGMRL pre-training

In [ ]:
def run_federated_ssl(client_splits, rounds=FED_ROUNDS, local_epochs=LOCAL_SSL_EPOCHS):
    global_model=make_model()
    global_teacher=copy.deepcopy(global_model).to(DEVICE)
    for p in global_teacher.parameters(): p.requires_grad_(False)
    history=[]
    for rnd in range(rounds):
        client_states=[]; client_weights=[]
        print(f"Federated SSL round {rnd+1}/{rounds}")
        for cname, splits in client_splits.items():
            local=copy.deepcopy(global_model).to(DEVICE)
            local_teacher=copy.deepcopy(global_teacher).to(DEVICE)
            loader=make_loader(splits["train"],train=True)
            opt=torch.optim.AdamW([p for p in local.parameters() if p.requires_grad],lr=2e-4,weight_decay=1e-4)
            losses=[]
            for ep in range(local_epochs):
                for images,_,_,_ in tqdm(loader,desc=f"{cname} SSL",leave=False):
                    images=images.to(DEVICE)
                    loss,_=dg_mrl_step(local,local_teacher,images,opt); losses.append(loss)
            # Only shared state leaves the client. Private adapters stay local.
            client_states.append(get_shared_state(local)); client_weights.append(len(splits["train"]))
            print(f"  {cname}: SSL loss={np.mean(losses) if losses else float('nan'):.4f}")
        aggregated=federated_average(client_states,client_weights)
        load_shared_state(global_model,aggregated)
        load_shared_state(global_teacher,aggregated)
        history.append({"round":rnd+1,"clients":len(client_states)})
    return global_model, global_teacher, history

# Run this cell after paths and metadata are configured.
# ssl_model, ssl_teacher, ssl_history = run_federated_ssl(CLIENT_SPLITS)

## 8. Supervised federated multi-label classification

In [ ]:
def masked_bce_loss(logits, labels, label_mask):
    raw=F.binary_cross_entropy_with_logits(logits,labels,reduction="none")
    return (raw*label_mask).sum()/(label_mask.sum()+1e-8)

def train_local_classifier(model, frame, epochs=SUPERVISED_EPOCHS, lr=1e-4, dp=True):
    model.train()
    loader=make_loader(frame,train=True)
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr,weight_decay=1e-4)
    for ep in range(epochs):
        losses=[]
        for batch in tqdm(loader,desc=f"supervised epoch {ep+1}/{epochs}",leave=False):
            if dp:
                patient_grouped_dp_step(model,opt,batch)
                images,labels,label_mask,_=batch
                with torch.no_grad():
                    logits=model(images.to(DEVICE),use_private=True)["logits"]
                    losses.append(float(masked_bce_loss(logits,labels.to(DEVICE),label_mask.to(DEVICE))))
            else:
                images,labels,label_mask,_=batch
                images=images.to(DEVICE); labels=labels.to(DEVICE); label_mask=label_mask.to(DEVICE)
                opt.zero_grad(set_to_none=True)
                loss=masked_bce_loss(model(images)["logits"],labels,label_mask)
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
                losses.append(float(loss.detach()))
        print(f"  epoch {ep+1}: masked BCE={np.mean(losses):.4f}" if losses else "  no batches")
    return model

def run_federated_supervised(initial_model, client_splits, rounds=1, epochs=SUPERVISED_EPOCHS, dp=True):
    global_model=copy.deepcopy(initial_model).to(DEVICE)
    for rnd in range(rounds):
        states=[]; weights=[]
        print(f"Federated supervised round {rnd+1}/{rounds}")
        for cname,splits in client_splits.items():
            local=copy.deepcopy(global_model).to(DEVICE)
            local=train_local_classifier(local,splits["train"],epochs=epochs,dp=dp)
            states.append(get_shared_state(local)); weights.append(len(splits["train"]))
        load_shared_state(global_model,federated_average(states,weights))
    return global_model

# Example:
# ssl_model, ssl_teacher, ssl_history = run_federated_ssl(CLIENT_SPLITS)
# final_model = run_federated_supervised(ssl_model, CLIENT_SPLITS, rounds=1, epochs=SUPERVISED_EPOCHS, dp=True)

## 9. Evaluation: finding-wise AUROC/AUPRC, sensitivity, specificity, F1, and calibration-ready outputs

In [ ]:
@torch.no_grad()
def predict_frame(model, frame, batch_size=BATCH_SIZE):
    model.eval(); loader=make_loader(frame,train=False,shuffle=False)
    ys=[]; ms=[]; probs=[]; pids=[]
    for images,labels,label_mask,patient_ids in tqdm(loader,desc="Evaluating",leave=False):
        out=model(images.to(DEVICE),use_private=False)["logits"]
        probs.append(torch.sigmoid(out).cpu().numpy())
        ys.append(labels.numpy()); ms.append(label_mask.numpy()); pids.extend(list(patient_ids))
    if not ys:
        return np.empty((0,len(LABELS))),np.empty((0,len(LABELS))),np.empty((0,len(LABELS))),[]
    return np.concatenate(ys),np.concatenate(ms),np.concatenate(probs),pids

def evaluate_multilabel(model, frame, threshold=0.5):
    y,mask,p,_=predict_frame(model,frame)
    rows=[]
    for j,lab in enumerate(LABELS):
        valid=mask[:,j].astype(bool)
        yt=y[valid,j].astype(int); pp=p[valid,j]
        if len(yt)==0:
            rows.append({"finding":lab,"n":0,"AUROC":np.nan,"AUPRC":np.nan,"sensitivity":np.nan,"specificity":np.nan,"F1":np.nan})
            continue
        pred=(pp>=threshold).astype(int)
        tp=np.sum((yt==1)&(pred==1)); tn=np.sum((yt==0)&(pred==0))
        fp=np.sum((yt==0)&(pred==1)); fn=np.sum((yt==1)&(pred==0))
        sens=tp/(tp+fn+1e-8); spec=tn/(tn+fp+1e-8)
        auc=roc_auc_score(yt,pp) if len(np.unique(yt))==2 else np.nan
        ap=average_precision_score(yt,pp) if len(np.unique(yt))==2 else np.nan
        rows.append({"finding":lab,"n":len(yt),"AUROC":auc,"AUPRC":ap,"sensitivity":sens,"specificity":spec,
                     "F1":f1_score(yt,pred,zero_division=0)})
    return pd.DataFrame(rows)

# After training, evaluate each institution's held-out patient split:
# for cname, splits in CLIENT_SPLITS.items():
#     print("\n", cname)
#     display(evaluate_multilabel(final_model, splits["test"]))

## 10. Limited-label experiment and ablation plan

In [ ]:
def make_limited_label_frame(train_frame, fraction=0.10, seed=SEED):
    # Select a fraction of PATIENTS, not images, to avoid leakage within a patient.
    patients=train_frame[PATIENT_COL].astype(str).drop_duplicates().to_numpy()
    rng=np.random.default_rng(seed); rng.shuffle(patients)
    keep=set(patients[:max(1,int(len(patients)*fraction))])
    return train_frame[train_frame[PATIENT_COL].astype(str).isin(keep)].copy().reset_index(drop=True)

# Example: 10% labels while SSL may still use all training images without labels.
# limited = {name: {**sp, "train": make_limited_label_frame(sp["train"], 0.10)}
#            for name,sp in CLIENT_SPLITS.items()}

ABLATIONS = {
    "full_DGFRL": {"disagreement_guided_masking": True, "srpa_routing": True, "private_adapters": True},
    "random_masking": {"disagreement_guided_masking": False, "srpa_routing": True, "private_adapters": True},
    "fixed_shared_private": {"disagreement_guided_masking": True, "srpa_routing": False, "private_adapters": True},
    "no_private_adapter": {"disagreement_guided_masking": True, "srpa_routing": True, "private_adapters": False},
}
print("Ablations declared:", list(ABLATIONS))
print("For manuscript results, train each setting under matched splits, rounds, seeds, and compute budgets.")

## 11. Save checkpoints and metrics

In [ ]:
OUTPUT_DIR = Path("./DGFRL_results")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

def save_run(model, metrics=None, name="dgfrl"):
    checkpoint=OUTPUT_DIR/f"{name}.pt"
    torch.save({"model_state_dict":model.state_dict(),"labels":LABELS,
                "config":{"img_size":IMG_SIZE,"patch_size":PATCH_SIZE,"embed_dim":EMBED_DIM,
                          "depth":DEPTH,"num_heads":NUM_HEADS,"lora_rank":LORA_RANK}},checkpoint)
    if metrics is not None: metrics.to_csv(OUTPUT_DIR/f"{name}_metrics.csv",index=False)
    print("Saved:",checkpoint.resolve())

def load_run(checkpoint_path):
    m=make_model()
    obj=torch.load(checkpoint_path,map_location=DEVICE)
    m.load_state_dict(obj["model_state_dict"])
    return m

# Example after training:
# all_metrics = []
# for cname, splits in CLIENT_SPLITS.items():
#     tmp=evaluate_multilabel(final_model,splits["test"]); tmp.insert(0,"client",cname); all_metrics.append(tmp)
# metrics_df=pd.concat(all_metrics,ignore_index=True)
# metrics_df.to_csv(OUTPUT_DIR/"institution_test_metrics.csv",index=False)
# save_run(final_model,metrics_df)

## 12. Suggested execution order

1. Install/import dependencies.
2. Set `DATASETS`, `LABELS`, and metadata column names.
3. Set `USE_SYNTHETIC_SMOKE_TEST=True` only if you want to verify the code without real data.
4. Run metadata loading and verify that patient IDs do not overlap across train/validation/test.
5. Run the model and DGMRL cells.
6. Run `ssl_model, ssl_teacher, ssl_history = run_federated_ssl(CLIENT_SPLITS)`.
7. Run `final_model = run_federated_supervised(ssl_model, CLIENT_SPLITS, rounds=1, epochs=SUPERVISED_EPOCHS, dp=True)`.
8. Evaluate test splits only after model selection is complete, then save results.

### Important implementation scope
- The state-space branch is a dependency-free **Mamba-inspired approximation**, not the official Mamba-2 kernel.
- The federated loop simulates institutions in one Python process; actual deployment requires each institution to run its local process and use a secure aggregation protocol.
- The patient-gradient routine clips the gradient of each patient's loss and adds noise, but rigorous DP claims require a vetted accountant (e.g., patient-level sampling assumptions and all shared trainable states must be included). Do not report an epsilon value without that accounting.
- The manuscript lists results as placeholders; this notebook computes metrics from your data and does not claim pre-existing performance.